# Mixed PDF 解析（图文混排类）

**适用**：有文本层的 PDF，不管是信息图、报告、杂志还是双栏。

**方法**：PyMuPDF 提取带坐标的文本块 → 按字体大小 + 位置分类。

**局限**：不识别特定版式（比如三列数据卡片），只做基础分类，够用于 RAG。

**依赖**：`pip install pymupdf`

In [3]:
from dataclasses import dataclass
from pathlib import Path

import fitz  # PyMuPDF


@dataclass
class ParsedChunk:
    content: str
    block_type: str  # title / paragraph / footer
    page: int


def _max_font_size(block: dict) -> float:
    """取一个块里出现的最大字号，用来判断是不是标题。"""
    sizes = [
        span["size"]
        for line in block.get("lines", [])
        for span in line.get("spans", [])
        if span.get("text", "").strip()
    ]
    return max(sizes) if sizes else 0



def parse_mixed_pdf(pdf_path: str | Path) -> list[ParsedChunk]:
    """
    解析 mixed 类型 PDF（有文本层的图文混排）。

    分类依据：
    - 字号较大（>= 正文均值 1.5 倍）→ title
    - 在页面底部 8% → footer
    - 其余 → paragraph
    """
    doc = fitz.open(Path(pdf_path))
    chunks: list[ParsedChunk] = []

    for page_index, page in enumerate(doc, start=1):
        page_height = page.rect.height

        # 只取文本块（type=0），跳过图片块（type=1）
        raw_blocks = [b for b in page.get_text("dict")["blocks"] if b["type"] == 0]

        # 按 y 坐标从上到下排序（阅读顺序）
        raw_blocks.sort(key=lambda b: b["bbox"][1])

        # 先算这一页的平均字号，作为"正文基准"
        all_sizes = [
            span["size"]
            for b in raw_blocks
            for line in b.get("lines", [])
            for span in line.get("spans", [])
            if span.get("text", "").strip()
        ]
        avg_size = sum(all_sizes) / len(all_sizes) if all_sizes else 12

        footer_parts: list[str] = []

        for block in raw_blocks:
            text = " ".join(
                span["text"]
                for line in block.get("lines", [])
                for span in line.get("spans", [])
            ).strip()

            if not text:
                continue

            # 块的垂直中心点
            y_center = (block["bbox"][1] + block["bbox"][3]) / 2

            if y_center > page_height * 0.92:
                # 位置在页面底部 → 页脚，先攒起来再合并
                footer_parts.append(text)
            elif _max_font_size(block) >= avg_size * 1.5:
                # 字号明显偏大 → 标题
                chunks.append(ParsedChunk(content=text, block_type="title", page=page_index))
            else:
                chunks.append(ParsedChunk(content=text, block_type="paragraph", page=page_index))

        if footer_parts:
            chunks.append(
                ParsedChunk(content=" ".join(footer_parts), block_type="footer", page=page_index)
            )

    doc.close()
    return chunks

## 测试 1：信息图页（极端 case）

`chevron-page.pdf` — 三列数据卡片 + 大图背景，版式很特殊，结果偏碎是正常现象。

In [ ]:
from pathlib import Path

# 测试：Chevron 信息图页
# 注意：这份 PDF 是极端信息图版式，全页都是打散的小字块。
# 通用解析器无法完美处理，结果比常规 mixed PDF（报告+插图）会碎一些，属于正常现象。
chunks = parse_mixed_pdf(Path("workshops/其他格式/pdf/chevron-page.pdf"))

print(f"共 {len(chunks)} 块\n")
print("=" * 60)
for i, chunk in enumerate(chunks, start=1):
    preview = chunk.content[:150] + "..." if len(chunk.content) > 150 else chunk.content
    print(f"[{i}] {chunk.block_type} (第{chunk.page}页)")
    print(f"    {preview}\n")

共 14 块

[1] paragraph (第1页)
    eastern mediterranean

[2] paragraph (第1页)
    We’re investing to grow in the Eastern Mediterranean, offshore Israel and Egypt. Natural gas production in the  region is helping to reduce Israel’s g...

[3] title (第1页)
    2.5 kilograms CO₂e/boe

[4] title (第1页)
    1st woman

[5] title (第1页)
    ~70%

[6] paragraph (第1页)
    electricity

[7] paragraph (第1页)
    carbon intensity from   our Eastern Mediterranean

[8] paragraph (第1页)
    offshore platform engineer   in Israel was employed by our

[9] paragraph (第1页)
    production in Israel is powered

[10] paragraph (第1页)
    by Tamar and Leviathan

[11] paragraph (第1页)
    operations in 2022

[12] paragraph (第1页)
    operations in 2020

[13] paragraph (第1页)
    fields as of 2022

[14] footer (第1页)
    Chevron 2022 Corporate Sustainability Report 1



## 测试 2：说明书 + 内嵌插图

`embedded-images.pdf` — 典型的 **mixed** 页：顶部示意图 + 下方说明段落。

比 Chevron 信息图更常见，解析效果通常更好；图内文字不在文本层里，提取不到。

In [5]:
chunks = parse_mixed_pdf(Path("workshops/其他格式/pdf/embedded-images.pdf"))

print(f"共 {len(chunks)} 块\n")
print("=" * 60)
for i, chunk in enumerate(chunks, start=1):
    preview = chunk.content[:150] + "..." if len(chunk.content) > 150 else chunk.content
    print(f"[{i}] {chunk.block_type} (第{chunk.page}页)")
    print(f"    {preview}\n")

共 13 块

[1] paragraph (第1页)
    “HANDS-FREE” CARD: use  (2/4)

[2] paragraph (第1页)
    5

[3] paragraph (第1页)
    3 4

[4] paragraph (第1页)
    Hands-free locking when   moving away from the   vehicle

[5] paragraph (第1页)
    Hands-free unlocking, when   approaching the vehicle;

[6] paragraph (第1页)
    If the card has been within the detec-  tion area  3  for approximately 15 min-  utes, remote locking is disabled. To lock  the vehicle, press the  5 ...

[7] paragraph (第1页)
    With the card in access zone  3 , the ve-  hicle will unlock. Unlocking is indicated  by  one flash  of the hazard warning  lights and the indicator l...

[8] paragraph (第1页)
    With the card on you, doors and lug-  gage compartment door closed, move  away from the vehicle: it will lock auto-  matically once you have left the ...

[9] paragraph (第1页)
    The vehicle cannot be remotely locked  if the card is within the area  4 .

[10] paragraph (第1页)
    Note:  the distance at which the vehicle  locks depends on